---
embed-resources: True
format:
    html:
        page-layout: full
        fontsize: 18px

title: 'Parte II | Markov y Monte Carlo'
author: 'Lorenzo Marquesini'
warning: False
toc: False
---

## Serie histórica

160 semanas para estimar la cadena. Columnas `period` y `state`, en el orden Normal, Pico, Disrupcion. La última semana es Disrupcion: de ahí arranca la simulación. La semilla es 20260816.


In [29]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.observaciones_markov import ESTADOS, SEMILLA, observaciones_historicas

DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"

observaciones = observaciones_historicas(seed=SEMILLA)
observaciones.to_csv(DATA / "markov_observations.csv", index=False)

destinations = pd.read_csv(DATA / "destinations.csv")
routes = pd.read_csv(DATA / "routes.csv")
origins = pd.read_csv(DATA / "origins.csv")

salidas = {estado: 0 for estado in ESTADOS}
for origen, _destino in zip(observaciones["state"], observaciones["state"].iloc[1:]):
    salidas[origen] += 1

assert list(observaciones.columns) == ["period", "state"]
assert len(observaciones) == 160
assert observaciones["state"].iloc[-1] == "Disrupcion"
assert all(salidas[estado] > 0 for estado in ESTADOS)

print("Semilla:", SEMILLA)
print("Último estado:", observaciones["state"].iloc[-1])
print("Salidas observadas:", salidas)
display(observaciones.head())
display(destinations)
display(routes.head())
display(origins)


Semilla: 20260816
Último estado: Disrupcion
Salidas observadas: {'Normal': 83, 'Pico': 66, 'Disrupcion': 10}


,period,state
0,0,Normal
1,1,Normal
2,2,Pico
3,3,Pico
4,4,Pico


,destination,nominal_demand,shortage_penalty,initial_inventory,storage_capacity,holding_cost,waste_cost
0,Capital,50.0,80.0,8.0,65.0,0.40,0.80
1,Oeste,42.0,70.0,6.0,55.0,0.35,0.70
2,Este,38.0,65.0,5.0,50.0,0.32,0.65
3,Costa,46.0,75.0,7.0,60.0,0.38,0.75
4,Interior,35.0,60.0,5.0,48.0,0.30,0.60


,route_id,origin,destination,unit_cost,capacity
0,N-CAP,Norte,Capital,4.0,42.0
1,N-OES,Norte,Oeste,6.0,28.0
2,N-EST,Norte,Este,5.0,30.0
3,N-COS,Norte,Costa,9.0,18.0
4,N-INT,Norte,Interior,7.0,26.0


,origin,supply
0,Norte,60.0
1,Centro,95.0
2,Sur,75.0


In [30]:
observaciones

,period,state
0,0,Normal
1,1,Normal
2,2,Pico
3,3,Pico
4,4,Pico
...,...,...
155,155,Pico
156,156,Normal
157,157,Normal
158,158,Disrupcion


## Cadena de Markov

Tres estados:
* Normal
* Pico
* Disrupción

In [31]:
from IPython.display import HTML

BLUE = "#1D24AB"
RED = "#C0392B"
INK = "#1F2430"
LINE = "#D5D8E2"
ETIQUETAS = {"Normal": "Normal", "Pico": "Pico", "Disrupcion": "Disrupción"}

def estimar_matriz(serie, estados=ESTADOS):
    origen = serie["state"].iloc[:-1].to_numpy()
    destino = serie["state"].iloc[1:].to_numpy()
    conteos = (
        pd.crosstab(origen, destino)
        .reindex(index=list(estados), columns=list(estados), fill_value=0)
        .astype(int)
    )
    salidas = conteos.sum(axis=1)
    vacias = salidas[salidas == 0].index.tolist()
    if vacias:
        raise RuntimeError(
            "Fila no estimable, sin salidas observadas: " + ", ".join(map(str, vacias))
        )
    matriz = conteos.div(salidas, axis=0)
    if (matriz < 0).any().any() or not (matriz.sum(axis=1) - 1).abs().max() < 1e-9:
        raise RuntimeError("La matriz estimada no es estocástica por filas")
    return conteos, matriz

conteos, matriz = estimar_matriz(observaciones)

def td(text, color=INK, weight="500", italic=False):
    style = (
        f"border:1px solid {LINE}; padding:8px 16px; text-align:center; background:#fff;"
        f"color:{color}; font-weight:{weight};"
    )
    if italic:
        style += "font-style:italic;"
    return f"<td style='{style}'>{text}</td>"

def th(text, color, italic=True):
    return td(text, color=color, weight="700", italic=italic).replace("td", "th", 2)

corner = (
    f"<th style='position:relative; min-width:118px; height:56px; border:1px solid {LINE};"
    "background:linear-gradient(to top right,#fff calc(50% - 0.7px),#C5C8D4 calc(50% - 0.7px),"
    f"#C5C8D4 calc(50% + 0.7px),#fff calc(50% + 0.7px));'>"
    f"<span style='position:absolute; top:6px; right:8px; color:{RED}; font-style:italic;'>Hacia</span>"
    f"<span style='position:absolute; bottom:6px; left:8px; color:{BLUE}; font-style:italic;'>Desde</span>"
    "</th>"
)
head = "".join(th(ETIQUETAS[estado], RED) for estado in matriz.columns)
body = []
for estado, fila in matriz.iterrows():
    celdas = th(ETIQUETAS[estado], BLUE) + "".join(td(f"{valor:.2f}") for valor in fila)
    body.append(f"<tr>{celdas}</tr>")

tabla = (
    "<table style='border-collapse:collapse; font-family:sans-serif; font-size:15px;'>"
    f"<thead><tr>{corner}{head}</tr></thead>"
    f"<tbody>{''.join(body)}</tbody></table>"
)
HTML(tabla)


HaciaDesde,Normal,Pico,Disrupción
Normal,0.72,0.24,0.04
Pico,0.24,0.68,0.08
Disrupción,0.60,0.10,0.30


### Simular posibles futuros

Una trayectoria tiene 26 semanas y empieza en Disrupción. El estado de la semana fija la media de la demanda, la probabilidad de que una ruta opere y el multiplicador de capacidad. Dado el estado, demandas y disponibilidades se sortean por separado.

#### 1. Las 26 semanas

La semana 1 es Disrupción, el último estado de la serie. De la semana 2 en adelante, el estado se sortea con la fila correspondiente de la matriz estimada.

In [32]:
import numpy as np

HORIZONTE = 26

def semanas(rng, matriz, horizonte=HORIZONTE, inicial="Disrupcion"):
    nombres = list(matriz.columns)
    estados = [inicial]
    for _ in range(horizonte - 1):
        fila = matriz.loc[estados[-1]].to_numpy(dtype=float)
        fila = fila / fila.sum()
        estados.append(nombres[int(rng.choice(len(nombres), p=fila))])
    return pd.DataFrame({"semana": np.arange(1, horizonte + 1), "estado": estados})

rng = np.random.default_rng(SEMILLA)
trayectoria = semanas(rng, matriz)
assert len(trayectoria) == HORIZONTE
assert trayectoria["estado"].iloc[0] == "Disrupcion"
trayectoria


,semana,estado
0,1,Disrupcion
1,2,Normal
2,3,Pico
3,4,Pico
4,5,Pico
5,6,Pico
6,7,Pico
7,8,Pico
8,9,Pico
9,10,Pico


#### 2. Demanda

En cada zona, la demanda de la semana es Poisson. La media es la demanda nominal multiplicada por el `demand_multiplier` del estado. Las cinco zonas se sortean independientes.

In [33]:
def demanda(rng, demanda_nominal, multiplicador):
    media = np.asarray(demanda_nominal, dtype=float) * multiplicador
    return rng.poisson(media)


#### 3. Disponibilidad

Cada ruta está disponible o no. El indicador es Bernoulli con probabilidad `route_availability` del estado. Las quince rutas se sortean independientes, también de la demanda.

In [34]:
def disponibilidad(rng, n_rutas, probabilidad):
    return (rng.random(n_rutas) < probabilidad).astype(int)


#### 4. Capacidad efectiva

No es otro sorteo. Si la ruta no está disponible, la capacidad es cero. Si lo está, la capacidad nominal se multiplica por el `capacity_multiplier` del estado.

In [35]:
def capacidad_efectiva(disponible, capacidad_nominal, multiplicador):
    operativo = np.asarray(disponible, dtype=float)
    nominal = np.asarray(capacidad_nominal, dtype=float)
    return operativo * nominal * multiplicador
